# R0 · Index cache

Builds the layout and all four indexes (Spindle, Spindle partial, Spindle-naive, Spindle-naive partial) of the 8
indexed sections for seeds 0–4 (10 % of the tiles held out at random), and stages the inputs every other notebook
reads, so that **no other notebook loads an h5ad or recomputes covariances**.

* **Prerequisites:** the datasets listed in `datasets.yaml`. Every other `R*` notebook needs this one.
* **Writes:** `results/_cache/<dataset>/` (the file list is in `experiments/common/data.py`).
* **Skips** every build and staging step whose outputs already exist under the current method tag (`FORCE = True` redoes all).
* **Builds:** `LOWMEM = True` (the default) runs the low-memory build (`experiments/common/lowmem_build.py`): the same
  files, bit for bit, with the per-tile matrices in scratch files on a local disk (system temp dir; brain_cancer needs
  about 18 GB free), in **< 6 GB of memory** for every dataset. `LOWMEM = False` runs the in-memory build
  (`index_cache.build`) whose stage times are Fig 2C and Supplementary Fig S3 (about 55 GB for brain_cancer, 30 GB for
  lymph_node_5k, ≤ 12 GB for the rest).
* **Slurm:** if `sbatch` is available, the low-memory builds run as one array with 16 GB per task; the in-memory builds
  run **one at a time on ROSALIND-CPU1**, because their stage times are Fig 2C and parallel builds on a shared node
  would contend. Staging then runs as an array of 8 tasks (16 GB each). Without Slurm, every step runs as its own
  Python process, in order.
* **Runtime / memory:** about 1.5 h of builds (one at a time) plus about 30 min of staging. Staging writes the
  covariances and whole-matrix vectors tile by tile, in < 6 GB of memory. With `LOWMEM = True`, nothing in the
  pipeline needs more than 16 GB.


In [1]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import pandas as pd

import data
import index_cache
import slurm

FORCE = False
LOWMEM = True  # False: the in-memory builds of Fig 2C (one at a time on ROSALIND-CPU1)
SCRIPT = "experiments/common/index_cache.py"
LOWMEM_SCRIPT = "experiments/common/lowmem_build.py"
print(data.stamp(), "| Slurm:", slurm.available())

{'tag': 'method-v2', 'describe': 'method-v2-dirty', 'commit': '8040f91'} | Slurm: True


## What is already there

In [2]:
st = index_cache.status()
st.pivot(index="key", columns="seed", values="built").assign(staged=st.groupby("key").staged.first())

seed,0,1,2,3,4,staged
key,,,,,,
brain_cancer,False,False,False,False,False,False
breast_cancer,False,False,False,False,False,False
kidney_nondiseased,False,False,False,False,False,False
lung_cancer,False,False,False,False,False,False
lymph_node,False,False,False,False,False,False
lymph_node_5k,False,False,False,False,False,False
pancreatic_cancer,False,False,False,False,False,False
skin_melanoma,False,False,False,False,False,False


## Build and stage what is missing

In [3]:
todo_build = [[ "build", r.key, str(r.seed)] for r in st.itertuples() if FORCE or not r.built]
todo_stage = [["stage", k] for k, g in st.groupby("key", sort=False) if FORCE or not g.staged.all() or not g.built.all()]
print(f"{len(todo_build)} builds, {len(todo_stage)} datasets to stage")
if LOWMEM:
    job_build = slurm.run(todo_build, LOWMEM_SCRIPT, sbatch="experiments/common/run.sbatch", name="R0_index_cache",
                          block=False, extra=["--mem=16G"])
else:
    job_build = slurm.run(todo_build, SCRIPT, sbatch="experiments/common/timing.sbatch", name="R0_index_cache",
                          serial=True, block=False)
job_stage = slurm.run(todo_stage, SCRIPT, sbatch="experiments/common/run.sbatch", name="R0_index_cache",
                      dependency=job_build or None, block=False, extra=["--mem=16G"])
if slurm.available() and (job_build or job_stage):
    slurm.wait([job_build, job_stage])
st = index_cache.status()
assert st.built.all() and st.staged.all(), st[~(st.built & st.staged)]

40 builds, 8 datasets to stage
submitted 24126: 40 x index_cache.py build (serial); logs results/_logs/R0_index_cache/
submitted 24127: 8 x index_cache.py stage (parallel); logs results/_logs/R0_index_cache/
  7021 s: 1 running, 0 pending   
all done in 7081 s


## Summary of the builds (shown, not saved; `R1_index_cost` writes `builds.csv`)
Storage of Spindle as % of Spindle-naive, and of Spindle partial as % of the Spindle-naive gene-set index; build seconds
exclude file loading.

In [4]:
rows = [json.loads((data.cache_dir(k, s) / "build.json").read_text()) for k in data.indexed_keys() for s in data.SEEDS]
b = pd.DataFrame(rows)
# peak memory: the low-memory builds' anonymous peak (their ru_maxrss also counts the mapped scratch files)
b["lowmem"] = b["lowmem"].fillna(False).astype(bool) if "lowmem" in b else False
b["peak"] = b.peak_anon_gb.fillna(b.peak_gb) if "peak_anon_gb" in b else b.peak_gb
b["build_s"] = b[["prepare_s", "layout_s", "naive_s", "spindle_s", "naive_partial_s", "spindle_partial_s"]].sum(axis=1)
g = b.groupby("key", sort=False)
pd.DataFrame({
    "cells": g.n_cells.first(), "genes": g.p.first(), "index tiles": g.n_train.median(), "held out": g.n_test.median(),
    "niches": g.n_niches.agg(lambda s: f"{s.min()}-{s.max()}"), "blocks": g.n_blocks.agg(lambda s: f"{s.min()}-{s.max()}"),
    "Spindle %": g.spindle_pct.mean().round(2), "Spindle partial %": g.spindle_partial_pct.mean().round(2),
    "build s": g.build_s.mean().round(1), "peak GB": g.peak.mean().round(1), "low-memory": g.lowmem.any(),
    "tags": g.tag.agg(lambda s: ",".join(sorted(set(s)))), "hosts": g.host.agg(lambda s: ",".join(sorted(set(s)))),
})

,cells,genes,index tiles,held out,niches,blocks,Spindle %,Spindle partial %,build s,peak GB,tags,hosts
key,,,,,,,,,,,,
skin_melanoma,87499,382,839.0,93.0,3-4,58-82,11.32,13.71,27.3,3.5,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
kidney_nondiseased,97560,377,705.0,78.0,2-3,40-60,11.26,14.34,23.8,3.0,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
breast_cancer,159226,313,1840.0,204.0,3-4,45-67,11.22,13.70,36.4,4.9,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
lung_cancer,162254,377,1535.0,170.0,3-4,57-77,11.20,13.70,36.9,5.8,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
pancreatic_cancer,190965,474,2061.0,229.0,4-4,108-111,11.97,14.49,62.6,11.5,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
lymph_node,377985,377,2354.0,261.0,5-7,103-146,10.84,13.94,52.0,8.6,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
lymph_node_5k,708983,400,6972.0,774.0,15-17,302-345,11.45,13.73,168.0,28.4,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu
brain_cancer,816769,480,9558.0,1062.0,16-18,418-480,11.56,14.14,259.7,52.8,method-v2,ROSALIND-CPU1.vuds.vanderbilt.edu


## Staged inputs on disk

In [5]:
def gb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e9

pd.DataFrame([{"key": k, "cache GB": round(gb(data.CACHE / k), 2),
               **{f: round((data.CACHE / k / f).stat().st_size / 1e9, 2) for f in ("covs.npy", "whole.npy", "cells.npz")},
               "stage peak GB": round((lambda r: r.get("peak_anon_gb", r["peak_gb"]))(
                   json.loads((data.CACHE / k / "whole.json").read_text())), 1)}
              for k in data.indexed_keys()])

,key,cache GB,covs.npy,whole.npy,cells.npz,stage peak GB
0,skin_melanoma,1.55,0.54,0.27,0.04,2.0
1,kidney_nondiseased,1.20,0.45,0.22,0.02,2.3
2,breast_cancer,2.39,0.80,0.40,0.09,2.6
3,lung_cancer,2.64,0.97,0.49,0.04,3.9
4,pancreatic_cancer,4.78,2.06,1.03,0.10,6.0
5,lymph_node,4.04,1.49,0.75,0.09,4.5
6,lymph_node_5k,14.26,4.96,2.48,0.47,11.6
7,brain_cancer,24.77,9.79,4.90,0.44,20.3
